In [1]:
import os
from concurrent.futures import ThreadPoolExecutor
import cv2 as cv
import pandas as pd
from pathlib import Path
import numpy as np
import shutil

In [2]:
raw = Path("../datasets/raw")
preprocessed = Path("../datasets/preprocessed")
df = pd.read_csv("../datasets/all.csv")

In [ ]:
def ap_lat_split(src : Path):
    files = src.glob('*')
    os.makedirs(src / "AP", exist_ok=True)
    os.makedirs(src / "LAT", exist_ok=True)
    for file in files:
        dst = "AP" if "AP" in file.stem else "LAT"
        shutil.move(file, src / dst)

### Image Preprocessing

In [3]:
# Processing Methods
def process_image(filename: str):
    image = cv.imread(filename, cv.IMREAD_GRAYSCALE)

    # crop
    border = image[0,0]
    mask = (image != border)
    non_border_mask = mask.astype(np.uint8) * 255
    coords = cv.findNonZero(non_border_mask)  # Returns x, y coordinates of non-black pixels

    if coords is not None:
        # Get the bounding box of the non-black region
        x, y, w, h = cv.boundingRect(coords)

        # Crop the image to the bounding box
        cropped = image[y:y+h, x:x+w]
        
    else:
        # If the image is entirely black, return the original image
        return image

    # resize
    resized = cv.resize(cropped, (1024, 1024))

    # CLAHE
    clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahed = clahe.apply(resized)

    # anything else

    return clahed

def process_and_save(files):
    for file in files:
        in_path = os.path.join(raw, file)
        out_path = os.path.join(preprocessed, file)

        try:
            processed = process_image(in_path) # processed := ndarray
            cv.imwrite(out_path, processed)
        except Exception as e:
            print(f"Failed to process {in_path}: {e}")
    
def split_work(files: list, pool: ThreadPoolExecutor):
        # Base case - pass images to thread.
        if len(files) <= 50:
            pool.submit(process_and_save, files)
        # Compute next split.
        else:
            split = len(files) // 2
            left = files[:split]
            right = files[split:]
            split_work(left, pool)
            split_work(right, pool)

In [4]:
# Processing Step
os.makedirs(preprocessed, exist_ok=True)

# Get list of image files
image_files = [f for f in os.listdir(raw) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

# Multithreading
with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    split_work(image_files, executor)

In [5]:
files = os.listdir(raw)

for file in files:
    if file not in df["ID"].unique():
        try:
            os.remove(preprocessed / file)
        except FileNotFoundError:
            continue

### train/test split

In [20]:
test_files = Path("../datasets/test")
train_files = Path("../datasets/train")
split = 0.2

count_dict = {col: np.floor(df[col].loc[df[col] == 1].count() * split) for col in df.columns[1:]}
print(count_dict)
id_list = set()

for _, row in df.iterrows():
    id = row['ID'].rsplit('_', 1)[0]
    add = True
    appearances = df[df['ID'].str.contains(id)]['ID'].to_numpy()
    
    
    for col in row.keys()[1:]:
        if row[col] == 1 and count_dict[col] < len(appearances):  # If adding would overflow the pathologies requirement
            add = False

    if add:
        for a in appearances:
            id_list.add(a)
            for col in row.keys()[1:]:
                if row[col] == 1:
                    count_dict[col] -= 1
len(id_list)

{'normal': np.float64(265.0), 'supracondylar fracture': np.float64(174.0), 'joint effusion': np.float64(123.0), 'soft tissue swelling': np.float64(61.0), 'lateral epicondyle displaced': np.float64(28.0)}


391

Move x-rays

In [61]:
shutil.rmtree(test_files)
shutil.rmtree(train_files)

os.makedirs(test_files, exist_ok=True)
os.makedirs(train_files, exist_ok=True)

cloned_df = df.copy()
# test imgs
for id in id_list:
    cloned_df.drop(cloned_df[cloned_df["ID"] == id].index, inplace=True)
    shutil.copy(preprocessed / id, test_files / id)

# train imgs
for col in cloned_df.columns[1:]:
    temp_df = cloned_df.where(cloned_df[col] == 1).dropna()
    os.makedirs(train_files / col, exist_ok=True)
    for name in temp_df["ID"]:
        shutil.copy(preprocessed / name, train_files / col / name)

for dir in train_files.iterdir():
    ap_lat_split(dir)

cloned_df.to_csv("../datasets/train.csv")

Move masks

In [ ]:
mask_dir = Path("../datasets/masks")
train_imgs = pd.read_csv("../datasets/train.csv")['ID'].to_numpy()
masks = mask_dir.rglob("*masklabel.png")
for m in masks:
    if 'normal' in m.parts: continue
    target = m.stem[:-10] + ".jpg"
    if target in train_imgs:
        dst = os.path.join(train_files, *m.parts[3:])
        shutil.copy(m, dst)

Normal split

In [86]:
old_normal = Path("../datasets/old_normal")
new_normal = Path("../datasets/train/normal")
full = set(new_normal.rglob("*.jpg"))

joint = set((old_normal / "joint").rglob("*masklabel.png"))
upper = set((old_normal / "upper").rglob("*masklabel.png"))
whole = set((old_normal / "whole").rglob("*masklabel.png"))

In [119]:
# joint iter
os.makedirs(new_normal / "AP" / "joint", exist_ok=True)
os.makedirs(new_normal / "LAT" / "joint", exist_ok=True)
for mask in joint:
    name = mask.stem[:-10]
    for img in full:
        name2 = img.stem
        if name == name2:
            dst = img.parent / "joint"
            shutil.copy(mask, dst)
            shutil.move(img, dst)

# upper iter
os.makedirs(new_normal / "AP" / "upper", exist_ok=True)
os.makedirs(new_normal / "LAT" / "upper", exist_ok=True)
for mask in upper:
    name = mask.stem[:-10]
    for img in full:
        name2 = img.stem
        if name == name2:
            dst = img.parent / "upper"
            shutil.copy(mask, dst)
            shutil.move(img, dst)

# whole iter
os.makedirs(new_normal / "AP" / "whole", exist_ok=True)
os.makedirs(new_normal / "LAT" / "whole", exist_ok=True)
for mask in whole:
    name = mask.stem[:-10]
    for img in full:
        name2 = img.stem
        if name == name2:
            dst = img.parent / "whole"
            shutil.copy(mask, dst)
            shutil.move(img, dst)